In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
from random import randint

In [2]:
url = "https://www.budgetbytes.com/category/recipes/main-dish/"

In [3]:
headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(url, headers=headers)
response.status_code # Check if the request was successful

200

In [4]:
# If the request was successful, parse the content
soup = BeautifulSoup(response.text, "html.parser")

In [5]:
# Find all recipe tiles on the page
recipe_tiles = soup.find_all('article', class_ = "post-summary")
len(recipe_tiles)

12

In [6]:
# Function to extract prep time, cook time, and servings from a recipe page
def get_info(link):
    response = requests.get(link, headers=headers)
    soup = BeautifulSoup(response.content, "html.parser")

    prep = soup.find("span", class_="wprm-recipe-prep_time-minutes")
    cook = soup.find("span", class_="wprm-recipe-cook_time-minutes")
    serv = soup.find("span", class_="wprm-recipe-servings")

    # Extract text if the elements are found, otherwise return None
    prep_time = prep.text if prep else None
    cook_time = cook.text if cook else None
    servings = serv.text if serv else None

    return prep_time, cook_time, servings 

In [7]:
# Create an empty list to store the data
rows = []

In [8]:
# Loop through each recipe tile and extract the required information
# This is only the first page, we will need to loop through multiple pages to get more data
for recipe in recipe_tiles:
    find_title = recipe.find('h3', class_ = "post-summary__title")
    find_cost = recipe.find('span', class_ = "cost-per")
    find_link = recipe.find('a')

    if find_title and find_cost and find_link:
        link = find_link['href']
        prep_time, cook_time, servings = get_info(link)

        rows.append({
            "title": find_title.text,
            "cost": find_cost.text,
            "prep_time": prep_time,
            "cook_time": cook_time,
            "servings": servings
        })  
    
    time.sleep(1)  # Sleep for 1 second to be polite to the server

# Check how many recipes we have collected
len(rows)

12

In [9]:
# Now we will loop through multiple pages to get more data
for page in range(2, 31):  # Scrape pages 2 to 30

    url = f"https://www.budgetbytes.com/category/recipes/main-dish/page/{page}/"

    r = requests.get(url, headers=headers)
    soup = BeautifulSoup(r.text, "html")

    recipe_tiles = soup.find_all('article', class_="post-summary")

    for recipe in recipe_tiles:
        find_title = recipe.find('h3', class_ = "post-summary__title")
        find_cost = recipe.find('span', class_ = "cost-per")
        find_link = recipe.find('a')

        if find_title and find_cost and find_link:
            link = find_link['href']
            prep_time, cook_time, servings = get_info(link)

            rows.append({
                "title": find_title.text,
                "cost": find_cost.text,
                "prep_time": prep_time,
                "cook_time": cook_time,
                "servings": servings
            })  
        time.sleep(randint(1, 3))  # Sleep for 1-3 seconds to be polite to the server
    time.sleep(randint(1, 3))  # Sleep for 1-3 seconds to be polite to the server

In [10]:
# Check how many recipes we have collected
len(rows)

357

In [11]:
# Create a DataFrame from the collected data
recipes = pd.DataFrame(rows)

In [12]:
# Preview the data
recipes

,title,cost,prep_time,cook_time,servings
0,Sliders,$11.44 recipe / $1.91 serving,10 minutes,25 minutes,6
1,Ham Tetrazzini,$7.65 recipe / $1.28 serving,15 minutes,45 minutes,6
2,Chicken Broth Bowl,$7.53 recipe / $1.89 serving,15 minutes,35 minutes,4
3,Feta Pasta,$11.05 recipe / $2.76 serving,20 minutes,40 minutes,4
4,Eggplant Curry,$7.32 recipe / $1.83 serving,15 minutes,45 minutes,4
...,...,...,...,...,...
352,Roasted Broccoli Pasta with Lemon and Feta,$3.69 recipe / $0.92 serving,15 minutes,25 minutes,4
353,BBQ Bean Sliders,$5.85 recipe / $1.46 serving,15 minutes,15 minutes,4
354,Smoky Roasted Sausage and Vegetables,$7.87 recipe / $1.97 serving,15 minutes,40 minutes,4
355,One Pot Spinach and Artichoke Pasta,$9.07 recipe / $1.62 serving,10 minutes,15 minutes,6


In [13]:
# Keep only unique recipes
recipes = recipes.drop_duplicates(subset='title')

In [14]:
# Reset the index after dropping duplicates
recipes.reset_index(drop=True, inplace=True)
recipes

,title,cost,prep_time,cook_time,servings
0,Sliders,$11.44 recipe / $1.91 serving,10 minutes,25 minutes,6
1,Ham Tetrazzini,$7.65 recipe / $1.28 serving,15 minutes,45 minutes,6
2,Chicken Broth Bowl,$7.53 recipe / $1.89 serving,15 minutes,35 minutes,4
3,Feta Pasta,$11.05 recipe / $2.76 serving,20 minutes,40 minutes,4
4,Eggplant Curry,$7.32 recipe / $1.83 serving,15 minutes,45 minutes,4
...,...,...,...,...,...
352,Roasted Broccoli Pasta with Lemon and Feta,$3.69 recipe / $0.92 serving,15 minutes,25 minutes,4
353,BBQ Bean Sliders,$5.85 recipe / $1.46 serving,15 minutes,15 minutes,4
354,Smoky Roasted Sausage and Vegetables,$7.87 recipe / $1.97 serving,15 minutes,40 minutes,4
355,One Pot Spinach and Artichoke Pasta,$9.07 recipe / $1.62 serving,10 minutes,15 minutes,6


In [15]:
# Split cost into total cost and cost per serving
recipes[['cost', 'cost_per_serving']] = recipes['cost'].str.split(' / ', expand=True)

In [16]:
recipes

,title,cost,prep_time,cook_time,servings,cost_per_serving
0,Sliders,$11.44 recipe,10 minutes,25 minutes,6,$1.91 serving
1,Ham Tetrazzini,$7.65 recipe,15 minutes,45 minutes,6,$1.28 serving
2,Chicken Broth Bowl,$7.53 recipe,15 minutes,35 minutes,4,$1.89 serving
3,Feta Pasta,$11.05 recipe,20 minutes,40 minutes,4,$2.76 serving
4,Eggplant Curry,$7.32 recipe,15 minutes,45 minutes,4,$1.83 serving
...,...,...,...,...,...,...
352,Roasted Broccoli Pasta with Lemon and Feta,$3.69 recipe,15 minutes,25 minutes,4,$0.92 serving
353,BBQ Bean Sliders,$5.85 recipe,15 minutes,15 minutes,4,$1.46 serving
354,Smoky Roasted Sausage and Vegetables,$7.87 recipe,15 minutes,40 minutes,4,$1.97 serving
355,One Pot Spinach and Artichoke Pasta,$9.07 recipe,10 minutes,15 minutes,6,$1.62 serving


In [17]:
# Fill missing cook_time and prep_time values with "0"
recipes['cook_time'] = recipes['cook_time'].fillna("0")
recipes['prep_time'] = recipes['prep_time'].fillna("0")

In [18]:
# Convert cost, cost_per_serving, prep_time, and cook_time to numeric values
recipes['cost'] = recipes['cost'].str.extract(r'(\d+\.\d+)').astype(float)
recipes['cost_per_serving'] = recipes['cost_per_serving'].str.extract(r'(\d+\.\d+)').astype(float)
recipes['prep_time'] = recipes['prep_time'].str.extract(r'(\d+)').astype(int)
recipes['cook_time'] = recipes['cook_time'].str.extract(r'(\d+)').astype(int)

In [19]:
recipes

,title,cost,prep_time,cook_time,servings,cost_per_serving
0,Sliders,11.44,10,25,6,1.91
1,Ham Tetrazzini,7.65,15,45,6,1.28
2,Chicken Broth Bowl,7.53,15,35,4,1.89
3,Feta Pasta,11.05,20,40,4,2.76
4,Eggplant Curry,7.32,15,45,4,1.83
...,...,...,...,...,...,...
352,Roasted Broccoli Pasta with Lemon and Feta,3.69,15,25,4,0.92
353,BBQ Bean Sliders,5.85,15,15,4,1.46
354,Smoky Roasted Sausage and Vegetables,7.87,15,40,4,1.97
355,One Pot Spinach and Artichoke Pasta,9.07,10,15,6,1.62


In [20]:
# Create a new column for total time by adding prep_time and cook_time
recipes['total_time'] = recipes['prep_time'] + recipes['cook_time']

In [21]:
recipes

,title,cost,prep_time,cook_time,servings,cost_per_serving,total_time
0,Sliders,11.44,10,25,6,1.91,35
1,Ham Tetrazzini,7.65,15,45,6,1.28,60
2,Chicken Broth Bowl,7.53,15,35,4,1.89,50
3,Feta Pasta,11.05,20,40,4,2.76,60
4,Eggplant Curry,7.32,15,45,4,1.83,60
...,...,...,...,...,...,...,...
352,Roasted Broccoli Pasta with Lemon and Feta,3.69,15,25,4,0.92,40
353,BBQ Bean Sliders,5.85,15,15,4,1.46,30
354,Smoky Roasted Sausage and Vegetables,7.87,15,40,4,1.97,55
355,One Pot Spinach and Artichoke Pasta,9.07,10,15,6,1.62,25


In [22]:
# Reorder the columns to have a cleaner look
recipes = recipes[
    ['title','cost', 'cost_per_serving', 'servings', 'prep_time', 'cook_time', 'total_time']
]

In [23]:
recipes

,title,cost,cost_per_serving,servings,prep_time,cook_time,total_time
0,Sliders,11.44,1.91,6,10,25,35
1,Ham Tetrazzini,7.65,1.28,6,15,45,60
2,Chicken Broth Bowl,7.53,1.89,4,15,35,50
3,Feta Pasta,11.05,2.76,4,20,40,60
4,Eggplant Curry,7.32,1.83,4,15,45,60
...,...,...,...,...,...,...,...
352,Roasted Broccoli Pasta with Lemon and Feta,3.69,0.92,4,15,25,40
353,BBQ Bean Sliders,5.85,1.46,4,15,15,30
354,Smoky Roasted Sausage and Vegetables,7.87,1.97,4,15,40,55
355,One Pot Spinach and Artichoke Pasta,9.07,1.62,6,10,15,25


In [53]:
recipes.to_csv("budgetbytes_recipes.csv", index=False)